# AdamWorld — generate a 3D world on a free Kaggle GPU

Turns a prompt or image into a Gaussian-splat world you can explore in the browser.
Runs the **default** pipeline on a single **T4 / P100 (~16 GB)**.

**Setup (one time):**
1. Enable the GPU: *Settings → Accelerator → GPU T4 x2* (one is used) or P100.
2. Enable internet: *Settings → Internet → On* (to download model weights).
3. Attach the `adamworld-core` dataset (the repo's `core/` folder) via *Add Input*,
   or run the fallback install cell.

The pipeline caches each stage by content hash under `/kaggle/working/world/cache`,
so if a session is cut off you can **Save & Run All** again and it resumes instead
of restarting.

In [ ]:
import os, glob, subprocess, sys

# AdamWorld source: attached as a Kaggle dataset (preferred) or cloned if you set GIT_URL.
SRC = next(iter(glob.glob("/kaggle/input/*/core")), None)
GIT_URL = ""  # optional: "https://github.com/<you>/adamworld.git"

if SRC is None and GIT_URL:
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, "/kaggle/working/adamworld"], check=True)
    SRC = "/kaggle/working/adamworld/core"
assert SRC and os.path.isdir(SRC), "Attach the adamworld-core dataset or set GIT_URL."
print("source:", SRC)

# Install adamworld's base (light) deps — NOT the pinned torch; Kaggle already ships torch.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", SRC], check=True)
# GPU model libraries (use Kaggle's preinstalled torch/CUDA).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.31.0", "transformers==4.46.2", "accelerate==1.1.1",
                "safetensors", "huggingface_hub", "opencv-python-headless"], check=True)
# gsplat builds against the installed torch/CUDA (prebuilt wheel if available).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gsplat"], check=True)
print("install done")

In [ ]:
import os, torch
os.environ["ADAMWORLD_CACHE"] = "/kaggle/working/.adamworld_cache"
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# --- parameters ---
PROMPT     = "a quiet forest clearing with a mossy stone well, soft morning light"
IMAGE      = None          # or "/kaggle/input/.../photo.jpg" to start from an image
RESOLUTION = 1024          # lower to 512/768 if you hit OOM
BACKEND    = "default"     # "default" (SDXL/FLUX/DepthAnything) | "flux-pano" (NC) | "mock"
SEED       = 0
OFFLOAD    = True          # sequential CPU offload — required to fit FLUX/SDXL on 16 GB

In [ ]:
from pathlib import Path
from adamworld.pipeline import Pipeline
from adamworld.profiling import Profiler
from adamworld.config import Settings
from adamworld.types import GenerationRequest

settings = Settings(offload=OFFLOAD)
prof = Profiler()
def progress(stage, frac):
    print(f"{frac*100:6.1f}%  {stage}", flush=True)

req = GenerationRequest(
    prompt=PROMPT,
    image_paths=[Path(IMAGE)] if IMAGE else [],
    resolution=RESOLUTION,
    backend=BACKEND,
    seed=SEED,
)
world = Pipeline(settings=settings, profiler=prof, progress=progress).generate(
    req, Path("/kaggle/working/world")
)
print("\n", prof.table(), sep="")
print("world:", world.root)

In [ ]:
import shutil
from IPython.display import Image as IPyImage, display

# Thumbnail + splat stats.
display(IPyImage(filename=str(world.thumbnail)))
from adamworld.splat import GaussianCloud
print("splats:", len(GaussianCloud.load_ply(world.splats_ply)))

# Zip the world for download (Output tab -> world.zip).
shutil.make_archive("/kaggle/working/world", "zip", world.root)
print("wrote /kaggle/working/world.zip  — download from the Output panel.")